# ML-09 — Validation and Research Claim Audit
Hicham Halfi · 2026-10-20

This audit reproduces Week 5, changes the split to hold out clients, and then removes overlapping-window features. It uses the bundled 30,000-row starter snapshot, not the 79M-row warehouse. No future deployment performance or causal effect is claimed.

## 1. Two paper findings and methodology questions
Source: [FlyRank, The State of AI-Driven SEO, March 2026](../../docs/flyrank-seo-research-march-2026.pdf).

**Page 27, health-score feature importance:** position and impressions lead the forest. The paper explicitly discloses that health is partly constructed from its inputs. Could the authors compare with an external outcome that is not a component of health, and report performance after excluding its component features? This would separate reconstruction of a scoring rule from independent predictive evidence. I retain the paper's descriptive framing.

**Page 29, growth classification:** the reported holdout accuracy is 71%. What is the class prevalence and majority-class accuracy on that same holdout; were clients separated, and were feature windows strictly earlier than the trend-label window? Grouped and forward-time validation would clarify transportability. The paper already labels these models exploratory; this question asks what further evidence would support operational use.

The paper's page 5 trend threshold is +/-10%, while the starter dictionary defines +/-20%. These populations and labels differ; the reported 71% is not a benchmark for this notebook.


In [1]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, GroupShuffleSplit, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/raw/content_refresh_anonymized.csv').exists())
OUT = ROOT/'work/outputs'; OUT.mkdir(exist_ok=True)
FIG = ROOT/'work/figures'; FIG.mkdir(exist_ok=True)
path = ROOT/'data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(path)
df = df.loc[df.impressions_90d > 0].reset_index(drop=True)
y = df.trend_direction.str.lower().eq('down').astype(int)
original = ['content_age_days','days_since_last_update','impressions_90d','avg_position','ctr']
prior = ['impressions_prev_30d','clicks_prev_30d','sessions_prev_30d']
assert df.content_id.is_unique
assert len(df) == 30000 and df.client_id.nunique() == 32
assert not set(original+prior) & {'trend_direction','trend_pct','client_id','content_id'}
print('Rows:', len(df), 'clients:', df.client_id.nunique(), 'positive rate:', round(y.mean(),4))
print('Version:', sklearn.__version__)


Rows: 30000 clients: 32 positive rate: 0.5421
Version: 1.9.1


## 2. Before/after split comparison
First reproduce the Week-5 forest (100 trees, depth 5, seed 42) and exact unstratified 80/20 split. Its original baseline is age >=180 and impressions >=500, ranked by impressions. Then use a client-grouped split: no client can occur on both sides. Architecture and feature set stay fixed so this comparison isolates the split change.

The third experiment uses only previous-30-day totals. Original 90-day aggregates overlap the most-recent-30-day label window; those features describe current state rather than support a forward prediction claim. The prior-window baseline ranks previous impressions only: snapshot age and freshness are deliberately excluded. Previous impressions also enter the label denominator, so this is still a retrospective classification exercise; it is not a sealed future test. Exact calendar dates are unavailable in this snapshot.


In [2]:
indices = np.arange(len(df))
random_train, random_test = train_test_split(indices, test_size=.2, random_state=42)
group_train, group_test = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42).split(df, y, df.client_id))
assert not set(df.iloc[group_train].client_id) & set(df.iloc[group_test].client_id)

def metrics(actual, scores):
    order = np.argsort(-np.asarray(scores), kind='stable')
    return {'base_rate':float(np.mean(actual)), 'precision_at_20':float(np.asarray(actual)[order[:20]].mean()),
            'precision_at_50':float(np.asarray(actual)[order[:50]].mean()),
            'average_precision':float(average_precision_score(actual,scores)),
            'roc_auc':float(roc_auc_score(actual,scores)),
            'majority_accuracy':float(max(np.mean(actual),1-np.mean(actual)))}

def run(name, train, test, features, baseline_kind):
    X = df[features].fillna(0)
    model = RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42,n_jobs=1)
    model.fit(X.iloc[train],y.iloc[train])
    prob = model.predict_proba(X.iloc[test])[:,1]
    frame = df.iloc[test]
    if baseline_kind == 'week5':
        base = ((frame.content_age_days>=180)&(frame.impressions_90d>=500)).astype(int)*frame.impressions_90d
    else:
        base = frame.impressions_prev_30d
    record = {'experiment':name,'train_rows':len(train),'test_rows':len(test),
        'train_clients':df.iloc[train].client_id.nunique(),'test_clients':df.iloc[test].client_id.nunique(),
        'overlapping_clients':len(set(df.iloc[train].client_id)&set(frame.client_id)),
        'features':features, 'model':metrics(y.iloc[test],prob),'baseline':metrics(y.iloc[test],base)}
    return record, model, prob

r0,m0,p0 = run('week5_random_reproduction',random_train,random_test,original,'week5')
r1,m1,p1 = run('same_features_client_holdout',group_train,group_test,original,'week5')
r2,m2,p2 = run('prior_window_client_holdout',group_train,group_test,prior,'prior')
results = [r0,r1,r2]
for r in results:
    print(r['experiment'], 'client_overlap=',r['overlapping_clients'])
    print('model:',r['model']); print('baseline:',r['baseline'])
print('Original feature importance:', dict(zip(original,m1.feature_importances_.round(4))))


week5_random_reproduction client_overlap= 31
model: {'base_rate': 0.5446666666666666, 'precision_at_20': 0.9, 'precision_at_50': 0.86, 'average_precision': 0.7296126216673372, 'roc_auc': 0.7188285714797737, 'majority_accuracy': 0.5446666666666666}
baseline: {'base_rate': 0.5446666666666666, 'precision_at_20': 0.65, 'precision_at_50': 0.46, 'average_precision': 0.5329989272674636, 'roc_auc': 0.49351894496703475, 'majority_accuracy': 0.5446666666666666}
same_features_client_holdout client_overlap= 0
model: {'base_rate': 0.5109524582184002, 'precision_at_20': 0.5, 'precision_at_50': 0.62, 'average_precision': 0.5988661947714709, 'roc_auc': 0.6020708273004797, 'majority_accuracy': 0.5109524582184002}
baseline: {'base_rate': 0.5109524582184002, 'precision_at_20': 0.35, 'precision_at_50': 0.38, 'average_precision': 0.4770937883977794, 'roc_auc': 0.46410821691005644, 'majority_accuracy': 0.5109524582184002}
prior_window_client_holdout client_overlap= 0
model: {'base_rate': 0.5109524582184002,

## 3. Leakage and robustness audit

| Candidate | Decision | Reason |
|---|---|---|
| trend_direction, trend_pct | Exclude | Label and direct label source |
| last-30-day metrics, 90-day metrics, snapshot CTR/position | Exclude from prior-window model | Contain or overlap outcome window |
| content_id, client_id | Group/join only | Pseudonyms are not predictors |
| health scores / product decision flags | Exclude | Existing decision or composite target reconstruction |
| snapshot age / update recency | Exclude from prior-window model | Historical value at prediction time not established |
| prev-30-day totals | Retain with caveat | Earlier window, but impressions enter relative-change denominator |

Week 5's five selected features have no direct label column, but that alone does not establish temporal safety. Position 0 also means missing, not top rank. The prior-window model avoids that ambiguous feature. Missing numeric inputs are filled with 0 for the reproduction; the cell below reports missingness and checks retained fields. Population inclusion impressions_90d>0 uses the outcome-containing window, so performance is conditional on active pages. Client grouping addresses repeated clients; it cannot repair these population and temporal limitations.

We run a deliberately invalid label-feature control to test the harness, then five grouped folds for sensitivity. Neither holdout nor folds are described as sealed: this is an exploratory audit with disclosed comparisons.


In [3]:
print('Selected-feature missing counts:',df[original+prior].isna().sum().to_dict())
assert df[prior].notna().all().all()
bad_train = y.iloc[group_train].to_numpy().reshape(-1,1)
bad_test = y.iloc[group_test].to_numpy().reshape(-1,1)
bad = DecisionTreeClassifier(max_depth=1,random_state=42).fit(bad_train,y.iloc[group_train])
bad_accuracy = accuracy_score(y.iloc[group_test],bad.predict(bad_test))
assert bad_accuracy == 1.0
print('INVALID label-feature control accuracy:',bad_accuracy,'(never used operationally)')
folds=[]
for i,(tr,te) in enumerate(GroupKFold(5).split(df,y,df.client_id),1):
    rec,_,_ = run('group_fold_'+str(i),tr,te,prior,'prior'); folds.append(rec)
print('Prior-window fold model P@20:',[r['model']['precision_at_20'] for r in folds])
print('Prior-window fold baseline P@20:',[r['baseline']['precision_at_20'] for r in folds])
errors = df.iloc[group_test][prior].copy()
errors['actual_decline']=y.iloc[group_test].to_numpy(); errors['score']=p2
top=errors.sort_values('score',ascending=False,kind='stable').head(20)
print('False positives in prior-window top 20 (no identifiers):')
print(top.loc[top.actual_decline.eq(0)].head(5).to_string(index=False))
print('Highest-ranked false negatives at score threshold .5 (no identifiers):')
print(errors.loc[errors.actual_decline.eq(1)&errors.score.lt(.5)].sort_values('score',ascending=False).head(5).to_string(index=False))


Selected-feature missing counts: {'content_age_days': 0, 'days_since_last_update': 0, 'impressions_90d': 0, 'avg_position': 0, 'ctr': 0, 'impressions_prev_30d': 0, 'clicks_prev_30d': 0, 'sessions_prev_30d': 0}
INVALID label-feature control accuracy: 1.0 (never used operationally)
Prior-window fold model P@20: [0.65, 0.9, 0.65, 1.0, 0.7]
Prior-window fold baseline P@20: [0.5, 0.5, 0.55, 0.3, 0.75]
False positives in prior-window top 20 (no identifiers):
 impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  actual_decline    score
                 2156                0                  0               0 0.810721
                 2001                0                  0               0 0.809847
                 1924                0                  0               0 0.809847
                 1608                0                  0               0 0.796166
Highest-ranked false negatives at score threshold .5 (no identifiers):
 impressions_prev_30d  clicks_prev_30d  sessions_prev_30d

## 4. Claim rewrite and interpretation
Week 5 said the forest would rank pages perfectly and described false positives as evergreen/seasonal. Neither claim is supported: the model made errors, and the snapshot does not identify seasonality or evergreen content. Its printed importances also ranked impressions first, not CTR.

Replacement: in this starter sample, a shallow forest produces a directional decline-review ranking. Its measured performance is reported beside a transparent baseline on the same test rows and beside prevalence. Results vary with client split and feature timing. False positives are non-declining labeled rows ranked highly; their cause is unknown. No statement about Google's ranking mechanism, causal refresh benefit, calibrated risk, future accuracy, or warehouse-scale generalization follows.

Top-k samples are small: one changed label moves P@20 by five percentage points. Fold ranges describe split sensitivity, not confidence intervals or an independent replication. Human review remains necessary.


In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
labels=['Random / original','Client / original','Client / prior window']
x=np.arange(3)
fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(x-.18,[r['baseline']['precision_at_20'] for r in results],.36,label='Rule baseline')
ax.bar(x+.18,[r['model']['precision_at_20'] for r in results],.36,label='Random forest')
ax.scatter(x,[r['model']['base_rate'] for r in results],color='black',marker='_',s=200,label='Test prevalence')
ax.set(xticks=x,xticklabels=labels,ylim=(0,1),ylabel='Precision at 20',title='Measured ranking performance on the starter snapshot')
ax.legend(); fig.tight_layout(); fig.savefig(FIG/'validation_comparison.png',dpi=180); plt.close(fig)
payload={'dataset_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'rows':len(df),'clients':df.client_id.nunique(),
 'python':platform.python_version(),'sklearn':sklearn.__version__,'experiments':results,'group_folds':folds,
 'invalid_leak_control_accuracy':bad_accuracy,'scope':'retrospective starter snapshot; not future validation'}
(OUT/'validation_metrics.json').write_text(json.dumps(payload,indent=2),encoding='utf-8')
print('Saved work/outputs/validation_metrics.json and work/figures/validation_comparison.png')


Saved work/outputs/validation_metrics.json and work/figures/validation_comparison.png


## Self-check
- Two exact paper findings and constructive questions: included, with page references.
- Week-5 reproduction, grouped before/after, and prior-window sensitivity: computed above.
- Direct leakage, overlapping windows, group overlap, missingness, population selection, and real errors: inspected.
- Numerical receipts and chart: generated by code; no dataset is committed.
- Execution status is recorded in saved cell outputs. Commit status is verified separately using Git.
- Remaining limitation: no dated forward panel or independent external validation in this analysis.
